In [265]:
import random
from typing import Callable

import pandas as pd
from pandas import DataFrame, Series

In [266]:
iris = pd.read_csv("../../data/iris.csv")

In [267]:
decisive_class = "variety"
features = ["sepal.length", "sepal.width", "petal.length", "petal.width"]

In [268]:
forward_variety_mapping = iris[decisive_class].mode()
reverse_variety_mapping = forward_variety_mapping.reset_index().set_index(decisive_class)["index"]

In [269]:
iris[decisive_class] = iris[decisive_class].map(reverse_variety_mapping)

In [270]:
def shuffle_dataset(data: DataFrame) -> None:
    for i in range(len(data)):
        random_id = random.randint(0, len(data) - 1)
        data.iloc[i], data.iloc[random_id] = data.iloc[random_id], data.iloc[i]


def split_dataset(data: DataFrame, percentage: float) -> tuple[DataFrame, DataFrame]:
    assert 0 <= percentage <= 1

    take = int(len(data) * percentage)
    return data[:take], data[take:]

In [271]:
shuffle_dataset(iris)
train_data, validation_data = split_dataset(iris, 0.7)

In [272]:
X_train, y_train = train_data[features], train_data[decisive_class]
X_validation, y_validation = validation_data[features], validation_data[decisive_class]

In [273]:
lower_bound, upper_bound = X_train.min(), X_train.max()

X_train_norm = (X_train - lower_bound) / (upper_bound - lower_bound)
X_validation_norm = (X_validation - lower_bound) / (upper_bound - lower_bound)

In [278]:
def manhattan_distance(data: DataFrame, centroid: Series) -> Series:
    return (data - centroid).abs().sum(axis=1)


def k_nearest(classified: DataFrame, labels: Series, unclassified: DataFrame, k: int,
              metric: Callable[[DataFrame, Series], Series]) -> Series:
    predictions = Series(0, index=unclassified.index)

    for point_id, point in unclassified.iterrows():
        distances = metric(classified, point)
        nearest = distances.nsmallest(k).index
        voting = labels.loc[nearest].mode()

        predictions.loc[point_id] = voting[0]

    return predictions

In [279]:
for k in [2, 3, 4]:
    y_predicted = k_nearest(X_train_norm, y_train, X_validation_norm, k, metric=manhattan_distance)

    valid_predictions = (y_predicted == y_validation).sum()
    total_predictions = len(y_predicted)

    accuracy = valid_predictions / total_predictions

    print(f"Dokładność dla k={k} wynosi {accuracy * 100:.2f}%")

Dokładność dla k=2 wynosi 91.11%
Dokładność dla k=3 wynosi 95.56%
Dokładność dla k=4 wynosi 93.33%
